# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU** (free) or **A100**
4. Click **Save**

Then click **Runtime → Run all** (Ctrl+F9 / Cmd+F9). Takes ~3 minutes.

---

# TunedAI Labs — Causal Reasoning Benchmark

Runs both models against the real **CLadder benchmark** — the same 10,112-question test used to establish the 96.96% score.

| Model | CLadder Score |
|---|---|
| Base Qwen 2.5-7B | ~62% |
| **TunedAI Labs Causal Model** | **96.96%** |

Questions use fictional variable names (yupt, jyka, kwox, etc.) — models cannot recall answers from pretraining. Requires actual causal reasoning.

This notebook runs a **500-question stratified sample** (~30 min on T4). Results should land near the published scores.

---

## Step 1 — Install packages

In [ ]:
import subprocess, sys, importlib

pkgs = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']
import_names = {'transformers': 'transformers', 'peft': 'peft', 'accelerate': 'accelerate',
                'bitsandbytes': 'bitsandbytes', 'openai': 'openai', 'anthropic': 'anthropic'}

for pkg in pkgs:
    try:
        importlib.import_module(import_names[pkg])
        print(f"  {pkg} already installed ✓")
    except ImportError:
        print(f"  installing {pkg}...", end=" ", flush=True)
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            capture_output=True, text=True, timeout=180)
        if r.returncode == 0:
            print("✓")
        else:
            print(f"FAILED\n{r.stderr[-200:]}")

print("\n✓ All packages ready.")

---
## Step 2 — API keys (optional)

Leave blank to skip GPT-4o and Claude columns — the local model comparison still runs.

In [ ]:
OPENAI_API_KEY    = ""   # paste OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste Anthropic key here (optional)

print("OpenAI key:",    "set" if OPENAI_API_KEY    else "not set (GPT-4o column will be skipped)")
print("Anthropic key:", "set" if ANTHROPIC_API_KEY else "not set (Claude column will be skipped)")

---
## Step 3 — Load models

Downloads Qwen 2.5-7B-Instruct + TunedAI Labs LoRA adapter from HuggingFace.
Takes ~90 seconds on T4.

In [ ]:
import torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, logging
from peft import PeftModel
import openai, anthropic

logging.set_verbosity_info()  # show download progress

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → set T4 GPU.")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.0f} GB")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓ Tokenizer ready")

if vram_gb >= 20:
    print("\nLoading base model float16 (downloading ~14GB, takes 3-5 min)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
else:
    print("\nLoading base model 8-bit (downloading ~14GB, takes 3-5 min)...")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("✓ Base model ready")

print("\nLoading TunedAI Labs adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓ Adapter ready")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

logging.set_verbosity_warning()  # quiet again for inference
print("\n✓ All models ready.")

---
## Step 4 — Verify adapter integrity

Checks SHA256 of the downloaded adapter against the known-good v2_e1 hash (96.96% CLadder).

In [ ]:
import hashlib, os, glob

# SHA256 of the correct v2_e1 adapter (96.96% CLadder, 9805/10112 correct)
KNOWN_SHA256 = "e0967ac8460870eab2f114379c4d4e9640934f944e294e115eec3f6bc67eba13"

# Find the cached adapter file (HuggingFace downloads to ~/.cache/huggingface)
cache_pattern = os.path.expanduser(
    "~/.cache/huggingface/hub/models--tunedailabs--causal-reasoning-qwen-7b/**/*.safetensors")
candidates = [f for f in glob.glob(cache_pattern, recursive=True)
              if not f.endswith(".incomplete")]

if not candidates:
    print("⚠ Adapter not found in cache — run Step 3 first.")
else:
    adapter_file = candidates[0]
    print(f"Verifying: {os.path.basename(adapter_file)} ({os.path.getsize(adapter_file)/1e6:.0f} MB)")
    print("Computing SHA256...", end=" ", flush=True)

    sha = hashlib.sha256()
    with open(adapter_file, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)
    computed = sha.hexdigest()

    if computed == KNOWN_SHA256:
        print("✓")
        print(f"\n✓ SHA256 MATCH — confirmed v2_e1 adapter (96.96% CLadder)")
        print(f"  {computed}")
    else:
        print("✗")
        print(f"\n✗ SHA256 MISMATCH — wrong adapter loaded")
        print(f"  Expected: {KNOWN_SHA256}")
        print(f"  Got:      {computed}")
        print("\nFix: Runtime → Disconnect and delete runtime, then start over.")
        raise RuntimeError("Wrong adapter. Results would be invalid.")

---
## Step 5 — Run benchmark

Two modes — toggle `USE_CLADDER` below:

- `USE_CLADDER = False` ← **default** — generates fresh questions the model has never seen. Correct answers computed from the numbers. Proves genuine causal reasoning.
- `USE_CLADDER = True` — uses the real CLadder dataset (model was trained on this distribution).

Takes ~3 min for 20 questions, ~20 min for 200.

In [ ]:
import json, os, random, urllib.request, torch

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_QUESTIONS  = 200
RANDOM_SEED  = 42
USE_CLADDER  = False       # False = fresh generated (never seen by model)
                           # True  = real CLadder dataset
RESULTS_FILE = "/content/results.json"
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM = "You are a causal reasoning expert. Answer YES or NO only."
MODE   = "cladder" if USE_CLADDER else "generated"

VARS = [
    "florp","wumf","ziff","quunt","brent","mirk","drex","veld","plox","snuv",
    "trake","yulf","grizz","blont","spuv","kornt","whelp","frix","zolt","blumb",
    "krenz","vusp","torb","wenf","glurt","plix","snorv","bwomp","cruft","zilth",
    "greff","polm","dweez","slunx","fruvv","kwelp","throx","blimp","snurt","grolt"
]

def gen_fresh(seed):
    rng = random.Random(seed)
    X, Y = rng.sample(VARS, 2)
    qt = rng.choice(['r1_more','r1_less','r1_overall','r2_up','r2_down','r3'])
    def pct(v): return f"{round(v*100)}%"

    if qt in ('r1_more','r1_less'):
        px=round(rng.uniform(0.2,0.8),2); py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"The overall probability of {X} is {pct(px)}. "
            f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        if qt=='r1_more': return dict(given=gi, q=f"Is {Y} more likely when observing {X}?", a='yes' if py1>py0 else 'no', rung=1)
        else:             return dict(given=gi, q=f"Is {Y} less likely when observing {X}?",  a='yes' if py1<py0 else 'no', rung=1)

    elif qt=='r1_overall':
        px=round(rng.uniform(0.2,0.8),2); py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        py=px*py1+(1-px)*py0
        gi=(f"The overall probability of {X} is {pct(px)}. "
            f"The probability of not {X} and {Y} is {pct(round((1-px)*py0,2))}. "
            f"The probability of {X} and {Y} is {pct(round(px*py1,2))}.")
        d=rng.choice(['more','less'])
        if d=='more': return dict(given=gi, q=f"Is {Y} more likely than not {Y} overall?", a='yes' if py>0.5 else 'no', rung=1)
        else:         return dict(given=gi, q=f"Is {Y} less likely than not {Y} overall?",  a='yes' if py<0.5 else 'no', rung=1)

    elif qt in ('r2_up','r2_down'):
        py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        if qt=='r2_up': return dict(given=gi, q=f"Will {X} increase the chance of {Y}?",  a='yes' if py1>py0 else 'no', rung=2)
        else:           return dict(given=gi, q=f"Will {X} decrease the chance of {Y}?",  a='yes' if py1<py0 else 'no', rung=2)

    else:
        py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        d=rng.choice(['more','less'])
        if d=='more': return dict(given=gi, q=f"For those who are {X}, would it be more likely to see {Y} if the individual was not {X}?", a='yes' if py0>py1 else 'no', rung=3)
        else:         return dict(given=gi, q=f"For those who are {X}, would it be less likely to see {Y} if the individual was not {X}?", a='yes' if py0<py1 else 'no', rung=3)

# ── Build question list ───────────────────────────────────────────────────────
if USE_CLADDER:
    path="/content/cladder-v1-questions.json"
    if not os.path.exists(path):
        print("Downloading CLadder...", end=" ", flush=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/causalNLP/cladder/main/data/cladder-v1-questions.json", path)
        print("✓")
    with open(path) as f: all_q=json.load(f)
    random.seed(RANDOM_SEED)
    buckets={}
    for q in all_q:
        k=(q.get('meta',{}).get('rung',0), q.get('answer','').lower()); buckets.setdefault(k,[]).append(q)
    qs=[]
    per=max(1, N_QUESTIONS//len(buckets))
    for k in sorted(buckets): qs+=random.sample(buckets[k], min(per,len(buckets[k])))
    random.shuffle(qs); qs=qs[:N_QUESTIONS]
    questions=[dict(id=q.get('question_id',str(i)),
                    given=q.get('given_info','') if isinstance(q.get('given_info',''),str) else ' '.join(f"{k}: {v}" for k,v in q['given_info'].items()),
                    q=q['question'], a=q['answer'].lower().strip(),
                    rung=q.get('meta',{}).get('rung','?')) for i,q in enumerate(qs)]
    print(f"CLadder: {len(questions)} real benchmark questions")
else:
    questions=[{**gen_fresh(RANDOM_SEED+i), 'id': f"gen_{RANDOM_SEED}_{i}"} for i in range(N_QUESTIONS)]
    print(f"Generated {len(questions)} fresh questions — model has never seen these")
    print("Answers computed mathematically from the given probabilities.")

# ── Load saved results (only if same mode) ────────────────────────────────────
saved_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f: saved = json.load(f)
    if saved.get("mode") == MODE:
        saved_results = saved.get("data", {})
        print(f"Resuming — {len(saved_results)} of {len(questions)} done.")
    else:
        print(f"Mode changed ({saved.get('mode')} → {MODE}) — starting fresh.")

# ── Run ───────────────────────────────────────────────────────────────────────
def _yn(text):
    t=text.strip().lower(); w=t.split(); f=w[0].rstrip(".,!:") if w else ""
    if f in ("yes","no"): return f
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    s=t[:60]
    if "yes" in s and "no" not in s: return "yes"
    if "no"  in s and "yes" not in s: return "no"
    return "?"

def ask(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(text,return_tensors="pt").to("cuda")
    with torch.no_grad(): out=tuned_model.generate(**inp,max_new_tokens=10,do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:],skip_special_tokens=True).strip()

results = dict(saved_results)
b_right=t_right=done=0

for i, q in enumerate(questions):
    key=q['id']; full_q=f"{q['given']}\n{q['q']}" if q['given'] else q['q']
    expected=q['a']; rung=q['rung']

    if key in results:
        r=results[key]
        b_right+=(r['b']==expected); t_right+=(r['t']==expected); done+=1
        print(f"[Q{i+1} R{rung} — Base {'✓' if r['b']==expected else '✗'} · TunedAI {'✓' if r['t']==expected else '✗'}]")
        continue

    b=ask(full_q,False); t=ask(full_q,True)
    by=_yn(b); ty=_yn(t); bok=by==expected; tok=ty==expected
    b_right+=bok; t_right+=tok; done+=1
    flag=" ★" if tok and not bok else (" ↓" if bok and not tok else "")
    print(f"Q{i+1:>3} R{rung} | exp:{expected.upper()} "
          f"base:{by.upper()} {'✓' if bok else '✗'} tuned:{ty.upper()} {'✓' if tok else '✗'}{flag}  "
          f"[Base {b_right}/{done}={100*b_right/done:.0f}%  TunedAI {t_right}/{done}={100*t_right/done:.0f}%]")
    # Store b/t (parsed), ba/ta (raw), a (expected), rung — self-contained for results cell
    results[key]={"b":by,"t":ty,"ba":b,"ta":t,"a":expected,"rung":rung}
    with open(RESULTS_FILE,"w") as f: json.dump({"mode":MODE,"data":results},f,indent=2)

label = "CLadder benchmark" if USE_CLADDER else "Fresh generated (never seen by model)"
print(f"\n{'='*60}\n{label} — {done} questions")
print(f"  Base Qwen 2.5-7B  : {b_right}/{done} = {100*b_right/done:.1f}%")
print(f"  TunedAI Labs ★    : {t_right}/{done} = {100*t_right/done:.1f}%")
print(f"  Gap               : {100*(t_right-b_right)/done:+.1f} pp")
if not USE_CLADDER:
    print(f"\n  ✓ Answers computed from math — not from training data.")
print(f"{'='*60}")

---
## Results

In [ ]:
import json, os
from collections import defaultdict

RESULTS_FILE = "/content/results.json"

if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the benchmark cell above first.")
else:
    with open(RESULTS_FILE) as f: saved = json.load(f)
    mode = saved.get("mode", "unknown")
    res  = saved.get("data", {})
    n    = len(res)

    if n == 0:
        print("Results file is empty — run the benchmark cell first.")
    else:
        b_correct = t_correct = 0
        rung_stats = defaultdict(lambda: {"b":0,"t":0,"n":0})

        for r in res.values():
            expected = r.get("a","?")
            rung     = r.get("rung","?")
            bok = r["b"] == expected
            tok = r["t"] == expected
            b_correct += bok
            t_correct += tok
            rung_stats[rung]["n"] += 1
            rung_stats[rung]["b"] += bok
            rung_stats[rung]["t"] += tok

        label = "CLadder benchmark" if mode == "cladder" else "Fresh generated (never seen by model)"
        SEP = "="*60
        print(SEP)
        print(f"{label} — {n} questions")
        print(SEP)
        print(f"  Base Qwen 2.5-7B  : {b_correct:3d}/{n} = {100*b_correct/n:.1f}%")
        print(f"  TunedAI Labs ★    : {t_correct:3d}/{n} = {100*t_correct/n:.1f}%")
        print(f"  Gap               : {100*(t_correct-b_correct)/n:+.1f} pp")
        print(SEP)
        print("\nBy Rung:")
        for rung in sorted(rung_stats.keys()):
            s = rung_stats[rung]
            print(f"  Rung {rung}: Base {s['b']}/{s['n']}={100*s['b']/s['n']:.0f}%  "
                  f"TunedAI {s['t']}/{s['n']}={100*s['t']/s['n']:.0f}%")
        if mode != "cladder":
            print(f"\n  ✓ Answers computed from math — not from training data.")

---
## Try Your Own Question

The model was trained on short structured causal questions. Use this template:

**Template:**
> In [setting], [Group A] [did X]. [Group B] [did not do X].
> [Group A result]. [Group B result].
> Does [X] cause [outcome]?

Replace the example below and run the cell.

In [ ]:
MY_QUESTION = """
Factory workers were split into two groups. Group A received ergonomic chairs. Group B kept standard chairs.
After 6 months, Group A reported 40% less back pain. Group B reported no change.
Does using an ergonomic chair cause a reduction in back pain?
"""

base_ans  = ask(MY_QUESTION.strip(), use_adapter=False)
tuned_ans = ask(MY_QUESTION.strip(), use_adapter=True)

print(f"Question: {MY_QUESTION.strip()}\n")
print(f"[ BASE QWEN 2.5-7B ]\n{base_ans}\n")
print(f"[ TUNEDAI LABS ★ ]\n{tuned_ans}")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)